In [37]:
import os
import h5py
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [20]:
BASE_DIR = Path("../Datensatz/data-transfer-student")

In [29]:
def build_wsi_cohort(cohort_dir: Path, n_bins: int = 4) -> pd.DataFrame:
    df_clin = pd.read_csv(
        cohort_dir / "clinical/clinical_survival_per_slide.csv")
    wsi_dir = cohort_dir / "wsi_features/conch_v15"
    available_wsi = {p.stem for p in wsi_dir.glob("*.h5")}

    df_clin = df_clin.dropna(subset=['survival_months', 'censorship']).copy()
    df_clin = df_clin[df_clin['survival_months'] > 0].copy()
    df_clin['slide_id'] = df_clin['slide_id'].str.replace(
        '.svs', '', regex=False)
    df_clin = df_clin[df_clin["slide_id"].isin(available_wsi)].copy()

    
    df_clin["event"] = 1 - df_clin["censorship"] # 1 = Todesfall, 0 = zensiert


    df_clin["dx_num"] = df_clin["slide_id"].apply(
        # extrahiert die DX-Nummer aus dem slide_id String. Wenn keine gefunden wird, wird 999 gesetzt
        lambda s: int(re.search(r"-DX(\d+)\.", s).group(1)
                      ) if re.search(r"-DX(\d+)\.", s) else 999
    )
    df_clin["n_patches"] = df_clin["slide_id"].apply(
        # fügt eine neue Spalte n_patches hinzu, die die Anzahl der Patches in der HDF5-Datei für jeden Slide zählt
        lambda s: h5py.File(wsi_dir / f"{s}.h5", "r")["features"].shape[0]
    )
    df_clin = (
        df_clin.sort_values(by=["case_id", "dx_num", "n_patches"], ascending=[
                            True, True, False])
        .drop_duplicates(subset=["case_id"], keep="first")
        # da wir nur einen Slide pro Patient wollen, wird hier nach case_id gruppiert und der Slide mit der niedrigsten DX-Nummer und den meisten Patches behalten
        .reset_index(drop=True)
    )

    # 4. Diskrete Survival-Bins (4 Quartile auf Basis der Todesfälle / uncensored)
    # Extrahiert die Überlebenszeiten der Todesfälle
    uncensored_times = df_clin.loc[df_clin["event"] == 1, "survival_months"]
    # Teilt die Überlebenszeiten der Todesfälle in n_bins Quartile ein und gibt die Bin-Grenzen zurück
    _, q_bins = pd.qcut(uncensored_times, q=n_bins, retbins=True, labels=False)
    q_bins[0] = -1e-5  # Setzt die untere Grenze des ersten Bins auf einen kleinen negativen Wert, um sicherzustellen, dass alle Überlebenszeiten in den Bins enthalten sind
    # Setzt die obere Grenze des letzten Bins auf den maximalen Überlebenszeitwert + einen kleinen positiven Wert
    q_bins[-1] = df_clin["survival_months"].max() + 1e-5
    df_clin["time_bin"] = pd.cut(
        df_clin["survival_months"], bins=q_bins, labels=False, include_lowest=True)
    # Kombinierte Klasse aus Zensierung und Zeit-Bin (0..7) für exakte Stratifizierung
    df_clin["strat_label"] = df_clin["censorship"].astype(
        int) * n_bins + df_clin["time_bin"].astype(int)
    return df_clin

In [ ]:
def cross_validate(df: pd.DataFrame, n_splits: int = 5) -> pd.DataFrame:

    # 5-Fold StratifiedGroupKFold direkt für jede Kohorte berechnen
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    df["fold"] = -1 # fügt eine neue Spalte "fold" hinzu, die standardmäßig auf -1 gesetzt ist. Diese Spalte wird später verwendet, um die Fold-Zuordnung für jeden Patienten zu speichern.

    for fold, (train_idx, val_idx) in enumerate(
        sgkf.split(X=df, y=df["strat_label"], groups=df["site"])
    ):
        df.loc[val_idx, "fold"] = fold # weist die Fold-Nummer für die Validierungsindizes in der "fold"-Spalte des DataFrames zu. Auf diese Weise wird jeder Patient einer bestimmten Fold-Nummer zugeordnet, basierend auf der Stratifizierung und Gruppierung, die durch StratifiedGroupKFold durchgeführt wurde.

    return df

In [31]:
def build_dataset(cohort_dir: Path, n_bins: int = 4, n_splits: int = 5) -> pd.DataFrame:
    df_clin = build_wsi_cohort(cohort_dir, n_bins=n_bins)
    df_clin = cross_validate(df_clin, n_splits=n_splits)
    return df_clin

In [36]:
dataset 

,case_id,slide_id,age,site,survival_months,censorship,is_female,oncotree_code,event,dx_num,n_patches,time_bin,strat_label,fold
0,TCGA-6D-AA2E,TCGA-6D-AA2E-01Z-00-DX1.81E7420E-9C8B-4076-A60...,68,6D,11.89,1,1.0,CCRCC,0,1,4074,1,5,1
1,TCGA-A3-3308,TCGA-A3-3308-01Z-00-DX1.bbd79ced-9fb9-4601-8ba...,77,A3,0.53,1,1.0,CCRCC,0,1,1768,0,4,4
2,TCGA-A3-3311,TCGA-A3-3311-01Z-00-DX1.9674290b-42c6-4107-9eb...,57,A3,39.13,0,0.0,CCRCC,1,1,1140,2,2,4
3,TCGA-A3-3313,TCGA-A3-3313-01Z-00-DX1.10f8cd5e-1c4f-4bb4-96f...,59,A3,24.15,0,0.0,CCRCC,1,1,2763,1,1,4
4,TCGA-A3-3316,TCGA-A3-3316-01Z-00-DX1.d6248cf8-7b02-4dce-8c9...,57,A3,49.05,1,0.0,CCRCC,0,1,1745,2,6,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,TCGA-G6-A8L8,TCGA-G6-A8L8-01Z-00-DX1.32E022BA-9959-4E2F-9D9...,62,G6,35.84,0,1.0,CCRCC,1,1,5198,2,2,3
339,TCGA-GK-A6C7,TCGA-GK-A6C7-01Z-00-DX1.291DEE77-CD13-4D8E-BA3...,76,GK,2.00,1,1.0,CCRCC,0,1,649,0,4,1
340,TCGA-MM-A563,TCGA-MM-A563-01Z-00-DX1.97A1F7D9-4156-4061-88B...,41,MM,19.42,1,0.0,CCRCC,0,1,3292,1,5,1
341,TCGA-MM-A564,TCGA-MM-A564-01Z-00-DX1.F8F976F5-500F-4B3F-BF0...,68,MM,19.94,1,0.0,CCRCC,0,1,7378,1,5,1


In [ ]:
class SurvivalWSIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, wsi_dir: Path, max_patches: int = None):
        self.df = df.reset_index(drop=True)
        self.wsi_dir = Path(wsi_dir)
        self.max_patches = max_patches

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        h5_path = self.wsi_dir / f"{row['slide_id']}.h5"
        
        with h5py.File(h5_path, 'r') as f:
            features = f['features'][:]  # [M, 768] (CONCH v1.5)

        # Optionales Subsampling nur im Training (spart VRAM und beschleunigt Epochen)
        if self.max_patches is not None and len(features) > self.max_patches:
            indices = np.random.choice(len(features), self.max_patches, replace=False)
            features = features[indices]

        features_tensor = torch.tensor(features, dtype=torch.float32)
        
        target = torch.tensor([row['time_bin'], row['censorship']], dtype=torch.long)
        
        survival_months = torch.tensor(row['survival_months'], dtype=torch.float32)
        censorship = torch.tensor(row['censorship'], dtype=torch.float32)

        return features_tensor, target, survival_months, censorship